<a href="https://colab.research.google.com/github/BigFoots625/IntroductionMachineLearningwithPython/blob/main/Chapter4_Representing_Data_and_Engineering_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Representing Data and Engineering Features**
Up to this point, we have assumed that our data is already formatted as a 2D array of continuous, floating-point numbers. However, real-world data is rarely this clean. It often contains text, categorical labels, and highly skewed distributions.

The process of determining how to best represent your data for a specific machine learning algorithm is known as **feature engineering**. It is often considered the most crucial step in the machine learning pipeline. In this chapter, we will cover categorical variables, binning, polynomial features, and automatic feature selection.

In [1]:
!pip install mglearn
import mglearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 9.7 MB/s eta 0:00:00


### **Theoretical Explanation: Categorical Variables (One-Hot Encoding)**
Many features do not represent continuous quantities but rather distinct categories (e.g., "Color": Red, Blue, Green). Machine learning models cannot interpret these text strings; they require numbers.

We cannot simply assign integers (Red=1, Blue=2, Green=3) because algorithms will interpret these as continuous values with an inherent order (meaning it would assume Blue is somehow "between" Red and Green).

To solve this, we use **One-Hot Encoding** (or dummy variables). This replaces a categorical feature with one or more new features that only take the values 0 and 1.
For a "Color" feature with three possible values, we create three new columns: `Color_Red`, `Color_Blue`, and `Color_Green`. If a data point is Red, the `Color_Red` column gets a 1, and the others get a 0.

In [2]:
import pandas as pd

# Create a sample dataframe with categorical data
data = pd.DataFrame({'Age': [25, 32, 47, 51],
                     'Salary': [50000, 68000, 120000, 105000],
                     'Degree': ['Bachelors', 'Masters', 'PhD', 'Bachelors']})

display(data)

# Use pandas get_dummies to perform one-hot encoding
data_dummies = pd.get_dummies(data)
print("\nData after One-Hot Encoding:")
display(data_dummies)

,Age,Salary,Degree
0,25,50000,Bachelors
1,32,68000,Masters
2,47,120000,PhD
3,51,105000,Bachelors



Data after One-Hot Encoding:


,Age,Salary,Degree_Bachelors,Degree_Masters,Degree_PhD
0,25,50000,True,False,False
1,32,68000,False,True,False
2,47,120000,False,False,True
3,51,105000,True,False,False


### **Theoretical Explanation: Numbers Can Encode Categoricals**
A common pitfall is assuming that just because a feature is stored as an integer, it should be treated as continuous. For example, a "Zip Code" might be stored as `90210`, but Zip Code `90211` is not mathematically "greater" in a way that implies a continuous relationship for predicting housing prices.

Similarly, a user might provide a "Rating" of 1 to 5. If we want the model to treat each rating as a distinct category rather than a continuous slope, we must explicitly convert the integer column to strings before applying `get_dummies`, so that the encoder recognizes it as categorical data.

In [3]:
# Create a dataframe where 'Class' is a categorical variable encoded as integers
demo_df = pd.DataFrame({'Integer_Feature': [0, 1, 2, 1],
                        'Categorical_Class': [1, 2, 3, 1]})

# By default, get_dummies ignores integers
print("Default get_dummies (ignores integers):\n", pd.get_dummies(demo_df))

# Convert the integer class to a string so it gets one-hot encoded
demo_df['Categorical_Class'] = demo_df['Categorical_Class'].astype(str)
print("\nget_dummies after converting to string:\n", pd.get_dummies(demo_df))

Default get_dummies (ignores integers):
    Integer_Feature  Categorical_Class
0                0                  1
1                1                  2
2                2                  3
3                1                  1

get_dummies after converting to string:
    Integer_Feature  Categorical_Class_1  Categorical_Class_2  \
0                0                 True                False   
1                1                False                 True   
2                2                False                False   
3                1                 True                False   

   Categorical_Class_3  
0                False  
1                False  
2                 True  
3                False  


### **Theoretical Explanation: Binning, Discretization, Linear Models, and Trees**
Linear models are incredibly fast, but they are limited because they can only model linear relationships. If the relationship between an input feature and the target is non-linear (e.g., a curve), a standard linear model will perform poorly.

We can make linear models more powerful on continuous data by **binning** (or discretizing) the feature. We partition the continuous input range into a fixed number of bins (intervals). A data point's continuous value is replaced by the category of the bin it falls into. We then one-hot encode these bins.

This forces the linear model to learn a constant value for each bin, effectively allowing it to model non-linear, piecewise-constant relationships. Note that tree-based models (like Random Forests) already learn to split data in this manner natively, so binning continuous features is generally only beneficial for linear models.

In [4]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.linear_model import LinearRegression

# Generate synthetic 1D wave data
X, y = mglearn.datasets.make_wave(n_samples=100)

# Discretize the continuous feature into 10 bins, and one-hot encode the output
kb = KBinsDiscretizer(n_bins=10, strategy='uniform', encode='onehot-dense')
kb.fit(X)
X_binned = kb.transform(X)

print(f"Original X shape: {X.shape}")
print(f"Binned X shape: {X_binned.shape}")

# A linear regression on the binned data will now learn a distinct coefficient for each bin
reg = LinearRegression().fit(X_binned, y)

Original X shape: (100, 1)
Binned X shape: (100, 10)


### **Theoretical Explanation: Interactions and Polynomials**
Another way to enrich a feature representation, particularly for linear models, is adding **interaction features** and **polynomial features**.

* **Interactions:** We can multiply features together. If we have binned data and a continuous feature, multiplying them allows the linear model to learn not just a different intercept for each bin, but a completely different *slope* for each bin.
* **Polynomials:** We can expand continuous features by taking their powers (e.g., $x^2, x^3$). If we have two features $a$ and $b$, a polynomial expansion of degree 2 will yield the features $1, a, b, a^2, b^2$, and $ab$.

This mathematically transforms a linear model into a polynomial regression model, allowing it to fit highly complex curves while still using the fast, underlying linear math.

In [5]:
from sklearn.preprocessing import PolynomialFeatures

# Add polynomial features up to degree 3 (x, x^2, x^3)
poly = PolynomialFeatures(degree=3, include_bias=False)
poly.fit(X)
X_poly = poly.transform(X)

print("Entries of X:\n", X[:5])
print("\nEntries of X_poly (x, x^2, x^3):\n", X_poly[:5])

# Train a linear model on the polynomial features
reg_poly = LinearRegression().fit(X_poly, y)

Entries of X:
 [[-0.75275929]
 [ 2.70428584]
 [ 1.39196365]
 [ 0.59195091]
 [-2.06388816]]

Entries of X_poly (x, x^2, x^3):
 [[-0.75275929  0.56664654 -0.42654845]
 [ 2.70428584  7.3131619  19.77688015]
 [ 1.39196365  1.93756281  2.697017  ]
 [ 0.59195091  0.35040587  0.20742307]
 [-2.06388816  4.25963433 -8.79140884]]


### **Theoretical Explanation: Univariate Non-linear Transformations**
Linear models and Neural Networks perform best when the features have a roughly Gaussian (bell curve) distribution. However, in reality, features like "number of purchases" or "website clicks" often have highly skewed, long-tailed distributions where most values are small, but a few are extremely large.

We can apply mathematical functions like $\log$ or $\exp$ to compress or expand these ranges. For integer count data, the transformation $\log(x + 1)$ is extremely common. It pulls the massive outliers closer to the mean, making the data look more Gaussian, which dramatically improves the performance of models sensitive to scale. Tree-based models, which only care about the ordering of values, are entirely unaffected by these monotonic transformations.

In [6]:
from sklearn.linear_model import Ridge

# Generate highly skewed Poisson-distributed data
rnd = np.random.RandomState(0)
X_org = rnd.normal(size=(1000, 3))
w = rnd.normal(size=3)
X_poisson = rnd.poisson(10 * np.exp(X_org))
y_poisson = np.dot(X_org, w)

# Apply a log transformation: log(x + 1) to handle zeros
X_trans = np.log(X_poisson + 1)

# Evaluating Ridge regression before and after transformation
X_train, X_test, y_train, y_test = train_test_split(X_poisson, y_poisson, random_state=0)
score_org = Ridge().fit(X_train, y_train).score(X_test, y_test)

X_train_trans, X_test_trans, y_train_trans, y_test_trans = train_test_split(X_trans, y_poisson, random_state=0)
score_trans = Ridge().fit(X_train_trans, y_train_trans).score(X_test_trans, y_test_trans)

print(f"Ridge score on original skewed data: {score_org:.3f}")
print(f"Ridge score on log-transformed data: {score_trans:.3f}")

Ridge score on original skewed data: 0.622
Ridge score on log-transformed data: 0.875


### **Theoretical Explanation: Automatic Feature Selection**
Adding interactions, polynomials, and dummies can quickly bloat our dataset to thousands of features. Having too many features leads to overfitting and incredibly slow training times. To reduce dimensionality, we use **Feature Selection** methods to automatically keep the most useful features and discard the rest.

The book outlines three primary strategies:
1.  **Univariate Statistics:** Computes the statistical significance (like ANOVA or f-tests) of each feature independently with the target variable. It is extremely fast because it ignores feature interactions. We use `SelectKBest` (to keep a specific number) or `SelectPercentile` (to keep a percentage).
2.  **Model-Based Selection:** Uses a supervised machine learning model (like a Random Forest or an L1-regularized Lasso model) to judge the importance of each feature. The model is trained on the full dataset, and features with low `feature_importances_` or near-zero coefficients are dropped. This is more robust as it captures interactions.
3.  **Iterative Selection (Recursive Feature Elimination - RFE):** Builds a model, identifies the single least important feature, removes it, and then completely rebuilds the model from scratch. It repeats this process until the desired number of features remains. It is the most accurate but by far the most computationally expensive.

In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectPercentile, SelectFromModel, RFE
from sklearn.ensemble import RandomForestClassifier

cancer = load_breast_cancer()

# Add pure noise features to the data to test our selection algorithms
rng = np.random.RandomState(42)
noise = rng.normal(size=(len(cancer.data), 50))
X_w_noise = np.hstack([cancer.data, noise])

X_train, X_test, y_train, y_test = train_test_split(X_w_noise, cancer.target, random_state=0, test_size=.5)

# 1. Univariate Statistics (Keep top 50%)
select_uni = SelectPercentile(percentile=50)
select_uni.fit(X_train, y_train)
X_train_uni = select_uni.transform(X_train)

# 2. Model-Based Selection (Using a Random Forest to pick features)
select_model = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold="median")
select_model.fit(X_train, y_train)
X_train_model = select_model.transform(X_train)

# 3. Iterative Selection (RFE)
select_rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42), n_features_to_select=40)
select_rfe.fit(X_train, y_train)
X_train_rfe = select_rfe.transform(X_train)

print(f"Original shape with noise: {X_train.shape}")
print(f"Shape after Univariate Selection: {X_train_uni.shape}")
print(f"Shape after Model-Based Selection: {X_train_model.shape}")
print(f"Shape after Iterative Selection (RFE): {X_train_rfe.shape}")

Original shape with noise: (284, 80)
Shape after Univariate Selection: (284, 40)
Shape after Model-Based Selection: (284, 40)
Shape after Iterative Selection (RFE): (284, 40)


### **Theoretical Explanation: Utilizing Expert Knowledge**
The final concept in feature engineering is the most human element: incorporating domain expertise.

Machine learning algorithms can only learn from the data provided. If a domain expert (like a doctor, mechanic, or financial analyst) knows that a specific combination of factors is a strong indicator of the target variable, you should engineer that feature explicitly. Sometimes, calculating a specific ratio based on business logic is far more powerful than hoping a neural network will discover that ratio implicitly. Feature engineering is where the art and science of data interact.

### **Chapter 4 Summary**
* **Objective:** We explored how to manipulate, transform, and select data to optimally present it to machine learning algorithms.
* **Key Concepts Covered:**
    * **Categoricals:** Encoding text strings using One-Hot Encoding (`pd.get_dummies`) and treating integer labels carefully.
    * **Transformations:** Binning to help linear models map non-linear data; Polynomial features to allow linear math to fit curves; Logarithmic transformations to normalize highly skewed continuous distributions.
    * **Selection:** Pruning features to prevent overfitting using Univariate Statistics (fastest), Model-Based Selection (balanced), and Recursive Feature Elimination (most thorough).
* **Takeaway:** A simple linear model with excellently engineered features will often outperform a highly complex, deep neural network trained on raw, unprocessed data.